In [1]:
!pip install tqdm boto3 requests regex sentencepiece sacremoses --quiet
!pip install transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 8.0 MB/s eta 0:00:00


In [2]:
from tqdm.auto import tqdm
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
import re

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import transformers
from transformers import BertModel, BertTokenizer, get_scheduler, set_seed, AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [4]:
# CONFIGURATION CLASS
@dataclass
class Config:
    data_path: str = "/content/drive/MyDrive/Dataset/Sentiment/Hotel_ABSA/Hotel_ABSA/Train.txt"
    test_path: str = "/content/drive/MyDrive/Dataset/Sentiment/Hotel_ABSA/Hotel_ABSA/Dev.txt"
    output_path: str = "./test.txt"
    batch_size: int = 16
    pin_memory: bool = True
    num_workers: int = 2
    seed: int = 42
    lr: float = 5e-5

In [5]:
config = Config()

In [6]:
set_seed(config.seed)

In [7]:
def readData(path):
    with open(path, 'r', encoding='utf-8') as file:
      content = file.read()
      lines = content.split('\n\n')
      data = [line.split('\n') for line in lines]
      df = pd.DataFrame(data, columns=['id', 'reviews', 'label'])
      df = df.drop(['id', 'label'], axis=1)
      def process_data(data):
          rows = []
          for entry in data:
              # Tách phần review và các cặp aspect-sentiment
              review, aspects = entry.split("\n", 1)
              # Tìm tất cả các cặp {aspect, sentiment}
              matches = re.findall(r"\{(.*?),(.*?)\}", aspects)

              # Tạo một hàng cho bảng
              row = {'review': review.strip()}
              for i, (aspect, sentiment) in enumerate(matches, start=1):
                  row[f'aspect{i}'] = aspect.strip()
                  row[f'sentiment{i}'] = sentiment.strip()

              rows.append(row)

          return rows

      # Xử lý dữ liệu
      processed_data = process_data(lines)
      dataset = pd.DataFrame(processed_data)
      dataset = dataset.drop(['review'], axis=1)
      dataset = pd.concat([df, dataset], axis=1)
      dataset = dataset.where(pd.notna(dataset), None)
    return dataset

def GenerateAspectCategoriesDict(df):
    aspectCategories = np.array(df['aspect1'])
    aspectCategories = np.concatenate((aspectCategories, np.array(df['aspect2'])))
    aspectCategories = np.concatenate((aspectCategories, np.array(df['aspect3'])))
    aspectCategories = np.concatenate((aspectCategories, np.array(df['aspect4'])))
    aspectCategories = np.concatenate((aspectCategories, np.array(df['aspect5'])))
    aspectCategories = aspectCategories[aspectCategories != None]

    aspectCategories = aspectCategories.astype(str)
    aspectCategories = np.unique(aspectCategories)

    aspectCategoriesDictIdxToCat = dict(enumerate(aspectCategories))
    aspectCategoriesDictCatToIdx = {}
    for i, j in enumerate(aspectCategories):
        aspectCategoriesDictCatToIdx[j] = i

    return aspectCategoriesDictCatToIdx, aspectCategoriesDictIdxToCat

def PolarityDict():
    polarityDictPolToIdx = {}
    polarityDictIdxToPol = {}
    polarity = ["positive", "neutral", "negative"]
    for i, j in enumerate(polarity):
        polarityDictPolToIdx[j] = i
        polarityDictIdxToPol[i] = j
    return polarityDictPolToIdx, polarityDictIdxToPol

def GenerateAspectPolarityVector(df):
    data = np.array(df)
    aspectDictCatToIdx, aspectDictIdxToCat = GenerateAspectCategoriesDict(df)
    print(aspectDictCatToIdx)
    polarityDictCatToIdx, polarityDictIdxToCat = PolarityDict()

    categoryVec = []
    for i in range(len(df)):
        vec = [0 for k in range(102)]
        for j in range(1, 10, 2):
            if data[i][j] == None:
                break
            temp1 = aspectDictCatToIdx[data[i][j]]
            temp2 = polarityDictCatToIdx[data[i][j+1]]
            vec[(temp1*3)+temp2] = 1
        categoryVec.append(vec)

    return categoryVec

In [8]:
df = pd.DataFrame(readData(config.data_path))
df["Aspect"] = GenerateAspectPolarityVector(df)


{'FACILITIES#CLEANLINESS': 0, 'FACILITIES#COMFORT': 1, 'FACILITIES#DESIGN&FEATURES': 2, 'FACILITIES#GENERAL': 3, 'FACILITIES#MISCELLANEOUS': 4, 'FACILITIES#PRICES': 5, 'FACILITIES#QUALITY': 6, 'FOOD&DRINKS#MISCELLANEOUS': 7, 'FOOD&DRINKS#PRICES': 8, 'FOOD&DRINKS#QUALITY': 9, 'FOOD&DRINKS#STYLE&OPTIONS': 10, 'HOTEL#CLEANLINESS': 11, 'HOTEL#COMFORT': 12, 'HOTEL#DESIGN&FEATURES': 13, 'HOTEL#GENERAL': 14, 'HOTEL#MISCELLANEOUS': 15, 'HOTEL#PRICES': 16, 'HOTEL#QUALITY': 17, 'LOCATION#GENERAL': 18, 'ROOMS#CLEANLINESS': 19, 'ROOMS#COMFORT': 20, 'ROOMS#DESIGN&FEATURES': 21, 'ROOMS#GENERAL': 22, 'ROOMS#MISCELLANEOUS': 23, 'ROOMS#PRICES': 24, 'ROOMS#QUALITY': 25, 'ROOM_AMENITIES#CLEANLINESS': 26, 'ROOM_AMENITIES#COMFORT': 27, 'ROOM_AMENITIES#DESIGN&FEATURES': 28, 'ROOM_AMENITIES#GENERAL': 29, 'ROOM_AMENITIES#MISCELLANEOUS': 30, 'ROOM_AMENITIES#PRICES': 31, 'ROOM_AMENITIES#QUALITY': 32, 'SERVICE#GENERAL': 33}


In [9]:
df.head()

,reviews,aspect1,sentiment1,aspect2,sentiment2,aspect3,sentiment3,aspect4,sentiment4,aspect5,sentiment5,aspect6,sentiment6,aspect7,sentiment7,aspect8,sentiment8,Aspect
0,Vừa qua tôi có dùng dịch vụ tại Khách Sạn TTC ...,HOTEL#GENERAL,positive,SERVICE#GENERAL,positive,None,None,None,None,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,Tuy nhiên buffet sáng ở đây không được ngon và...,FOOD&DRINKS#QUALITY,negative,FOOD&DRINKS#STYLE&OPTIONS,negative,None,None,None,None,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,"Nhìn chung dịch vụ khách sạn cũng tốt, chỉ có ...",FOOD&DRINKS#QUALITY,negative,SERVICE#GENERAL,positive,None,None,None,None,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,"Đội ngũ nhân viên phục vụ chu đáo, tận tình.",SERVICE#GENERAL,positive,None,None,None,None,None,None,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,"Trang thiết bị nội thất hài hoà, tiện nghi hiệ...",ROOM_AMENITIES#DESIGN&FEATURES,positive,ROOM_AMENITIES#QUALITY,positive,HOTEL#QUALITY,positive,None,None,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [10]:
aspect_conversion_dict = {'FACILITIES#CLEANLINESS': 0, 'FACILITIES#COMFORT': 1, 'FACILITIES#DESIGN&FEATURES': 2,
                          'FACILITIES#GENERAL': 3, 'FACILITIES#MISCELLANEOUS': 4, 'FACILITIES#PRICES': 5, 'FACILITIES#QUALITY': 6,
                          'FOOD&DRINKS#MISCELLANEOUS': 7, 'FOOD&DRINKS#PRICES': 8, 'FOOD&DRINKS#QUALITY': 9,
                          'FOOD&DRINKS#STYLE&OPTIONS': 10, 'HOTEL#CLEANLINESS': 11, 'HOTEL#COMFORT': 12, 'HOTEL#DESIGN&FEATURES': 13,
                          'HOTEL#GENERAL': 14, 'HOTEL#MISCELLANEOUS': 15, 'HOTEL#PRICES': 16, 'HOTEL#QUALITY': 17,
                          'LOCATION#GENERAL': 18, 'ROOMS#CLEANLINESS': 19, 'ROOMS#COMFORT': 20, 'ROOMS#DESIGN&FEATURES': 21,
                          'ROOMS#GENERAL': 22, 'ROOMS#MISCELLANEOUS': 23, 'ROOMS#PRICES': 24, 'ROOMS#QUALITY': 25,
                          'ROOM_AMENITIES#CLEANLINESS': 26, 'ROOM_AMENITIES#COMFORT': 27, 'ROOM_AMENITIES#DESIGN&FEATURES': 28,
                          'ROOM_AMENITIES#GENERAL': 29, 'ROOM_AMENITIES#MISCELLANEOUS': 30, 'ROOM_AMENITIES#PRICES': 31,
                          'ROOM_AMENITIES#QUALITY': 32, 'SERVICE#GENERAL': 33}
sentiment_conversion_dict = {'positive': 2, 'neutral': 1, 'negative': 0}

df['aspect1'] = df['aspect1'].apply(lambda x: aspect_conversion_dict[x])
df['sentiment1'] = df['sentiment1'].apply(lambda x: sentiment_conversion_dict[x])

df['aspect2'] = df['aspect2'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
df['sentiment2'] = df['sentiment2'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

# df['aspect3'] = df['aspect3'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
# df['sentiment3'] = df['sentiment3'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

# df['aspect4'] = df['aspect4'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
# df['sentiment4'] = df['sentiment4'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

# df['aspect5'] = df['aspect5'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
# df['sentiment5'] = df['sentiment5'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)


In [11]:
a1 = df['aspect1']
s1 = df['sentiment1']
a1 += 1
s1 += 1
df['labels1'] = a1*s1 - 1

a2 = df['aspect2']
s2 = df['sentiment2']
a2 += 1
s2 += 1
df['labels2'] = a2*s2 - 1

df.head()

,reviews,aspect1,sentiment1,aspect2,sentiment2,aspect3,sentiment3,aspect4,sentiment4,aspect5,sentiment5,aspect6,sentiment6,aspect7,sentiment7,aspect8,sentiment8,Aspect,labels1,labels2
0,Vừa qua tôi có dùng dịch vụ tại Khách Sạn TTC ...,15,3,34.0,3.0,None,None,None,None,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",44,101.0
1,Tuy nhiên buffet sáng ở đây không được ngon và...,10,1,11.0,1.0,None,None,None,None,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",9,10.0
2,"Nhìn chung dịch vụ khách sạn cũng tốt, chỉ có ...",10,1,34.0,3.0,None,None,None,None,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",9,101.0
3,"Đội ngũ nhân viên phục vụ chu đáo, tận tình.",34,3,NaN,NaN,None,None,None,None,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",101,NaN
4,"Trang thiết bị nội thất hài hoà, tiện nghi hiệ...",29,3,33.0,3.0,HOTEL#QUALITY,positive,None,None,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",86,98.0


In [13]:
classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(768, 102), nn.Softmax())

In [14]:
class BertForABSA(nn.Module):
    def __init__(self, bert, num_labels=102):
        super(BertForABSA, self).__init__()
        self.bert = bert
        self.classifier = classifier
#         self.dropout = nn.Dropout()
#         self.classifier = nn.Linear(768, num_labels)
#         self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask):
        _, pooled_output = self.bert(input_ids, attention_mask,
                                     return_dict=False)

        logits = self.classifier(pooled_output)
#         pooled_output = self.dropout(pooled_output)
#         logits = self.classifier(pooled_output)
#         logits = self.relu(logits)

        return logits

In [15]:
class ReviewsDataset(Dataset):
    def __init__(self, train_data, tokenizer, label_col, max_sequence_len=120, as_float=False):
        self.as_float = as_float
        print("Starting Process ...")
        labels = list(train_data[label_col].values)
        # Number of exmaples.
        self.n_examples = len(labels)
        # Use tokenizer on texts. This can take a while.
        print('Using tokenizer on all texts ...')

        texts = list(train_data['reviews'].values)
        self.inputs = tokenizer(texts, add_special_tokens=False, \
                                truncation=True, padding=True, \
                                return_tensors='pt')

        # Get maximum sequence length.
        self.sequence_len = self.inputs['input_ids'].shape[-1]
        print('Texts padded or truncated to %d length!' % self.sequence_len)
        # Add labels.
        self.labels = torch.tensor(labels)
        print('Finished!\n')

    def __len__(self):
        return self.n_examples

    def __getitem__(self, i):
        if self.as_float:
            return {key: self.inputs[key][i] for key in self.inputs.keys()}, self.labels[i].to(torch.float)
        else:
            return {key: self.inputs[key][i] for key in self.inputs.keys()}, self.labels[i]

In [16]:
bert = AutoModel.from_pretrained("vinai/phobert-base")
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
# tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# bert = BertModel.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

In [17]:
dataset = ReviewsDataset(df, tokenizer, 'labels1')

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Starting Process ...
Using tokenizer on all texts ...
Texts padded or truncated to 244 length!
Finished!



In [18]:
inputs, labels = dataset[0]
print(inputs, labels)

{'input_ids': tensor([ 4087,    89,    70,    10,   175,  1626,   178,    35,  5432,   870,
        14614, 23585, 20001, 25383,  2265,  2507,  2265,  2507,  5194, 30676,
        13208,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1, 

In [19]:
# config.train_size = len(dataset) - config.val_size
train_ds, val_ds = random_split(dataset, [1/9,8/9])

In [20]:
train_loader = DataLoader(train_ds, config.batch_size, shuffle=True,
                          num_workers=config.num_workers,
                          pin_memory=config.pin_memory)

val_loader = DataLoader(val_ds, config.batch_size, shuffle=False,
                        num_workers=config.num_workers,
                        pin_memory=config.pin_memory)

In [21]:
for input, label in train_loader:
    print(input)
    print()
    print(label)
    break

{'input_ids': tensor([[  218,   480,   133,  ...,     1,     1,     1],
        [ 7794,  2608,   915,  ...,     1,     1,     1],
        [ 1584,    94,  8840,  ...,     1,     1,     1],
        ...,
        [ 4087, 38937,     4,  ...,     1,     1,     1],
        [ 3244,    10,   143,  ...,     1,     1,     1],
        [ 5432, 20014,  1187,  ...,     1,     1,     1]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

tensor([ 44, 101,  50,  56, 101,  89,  56,  44,  89,  44,  33, 101,  38, 101,
         30,  56])


In [22]:
def get_one_hot(outputs, k=2):
    outputs = outputs.detach()
    x = torch.topk(outputs, k)
    for i, t in enumerate(outputs):
#         x = torch.topk(t, k[i])
        for j, _ in enumerate(t):
            if j in x.indices[i]:
                t[j] = 1
            else:
                t[j] = 0
            outputs[i] = t

    outputs.requires_grad = True
    return outputs.to(torch.float)

def get_accuracy(outputs, labels):
    preds = torch.argmax(outputs, dim=1)
    return (preds == labels).sum()

def one_hot_acc(one_hot_outputs, labels):
    result = torch.all(one_hot_outputs.eq(labels))
    return result.sum()

In [23]:
model = BertForABSA(bert)

In [24]:
def train(model, train_dataloader, val_dataloader, learning_rate, epochs):
    # track the time and history
    start = perf_counter()
    history = []
    # check for cuda use
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    if use_cuda:
        torch.cuda.empty_cache()

    # prepare optimizer, loss-criterion and lr-scheduler
#     criterion = nn.BCELoss()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)#, weight_decay=0.0001)
    num_training_steps = epochs * len(train_dataloader)
    lr_scheduler = get_scheduler(name="linear", optimizer=optimizer,
                                 num_warmup_steps=0,
                                 num_training_steps=num_training_steps)

    # prepare a progress bar
    progress_bar = tqdm(range(num_training_steps))

    # move model to the GPU
    if use_cuda:
            # linear = linear.cuda()
            model = model.cuda()
            criterion = criterion.cuda()

    # Start Epoch wise training
    for epoch_num in range(epochs):
        epoch_start = perf_counter()
        total_acc_train = 0
        total_loss_train = 0

        # Training Phase
        model.train()
        for inputs, label in train_dataloader:
            label = label.to(torch.long)
            label = label.to(device)
            # label = torch.argmax(label, dim=1)

            # max_token_type_id = model.bert.embeddings.token_type_embeddings.num_embeddings - 1
            # inputs['token_type_ids'] = torch.clamp(inputs['token_type_ids'], 0, max_token_type_id)

            inputs['attention_mask'] = inputs['attention_mask'].to(device)
            inputs['input_ids'] = inputs['input_ids'].to(device)
            # inputs['token_type_ids'] = inputs['token_type_ids'].to(device)
            inputs.pop("token_type_ids", None)

            outputs = model(**inputs)
            ## k = label.sum(dim=1).to(torch.int)
            ## assert len(k) == config.batch_size
#           ##  one_hot_outputs = get_one_hot(outputs)

            # loss calculation step
            batch_loss = criterion(outputs, label)
            # batch_loss = criterion(one_hot_outputs, label)
            batch_loss.backward()

            # collate losses and acc.
            total_loss_train += batch_loss.item()
            acc = get_accuracy(outputs, label)
            # acc = one_hot_acc(one_hot_outputs, label)
            total_acc_train += acc

            # Update step
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

        # Validation Phase
        total_acc_val = 0
        total_loss_val = 0
        model.eval()
        with torch.no_grad():
            for val_input, val_label in val_dataloader:
                val_label = val_label.to(torch.long)
                val_label = val_label.to(device)
#                 val_label = torch.argmax(val_label, dim=1)

                val_input['attention_mask'] = val_input['attention_mask'].to(device)
                val_input['input_ids'] = val_input['input_ids'].to(device)
                # val_input['token_type_ids'] = val_input['token_type_ids'].to(device)
                val_input.pop("token_type_ids", None)

                outputs = model(**val_input)
#                 k = label.sum(dim=1).to(torch.int)
#                 assert len(k) == config.batch_size
#                 one_hot_outputs = get_one_hot(outputs)

                batch_loss = criterion(outputs, val_label)
#                 batch_loss = criterion(one_hot_outputs, val_label)
                total_loss_val += batch_loss.item()

                acc = get_accuracy(outputs, val_label)
#                 acc = one_hot_acc(one_hot_outputs, label)
                total_acc_val += acc

        # measure epoch-time
        epoch_time = perf_counter() - epoch_start

        # print results
        print(f'\nEpochs: {epoch_num + 1}/{epochs} | Train Loss: {total_loss_train / len(train_ds): .3f} \
| Train Accuracy: {total_acc_train / len(train_ds): .3f} | Val Loss: {total_loss_val / len(val_ds): .3f} \
| Val Accuracy: {total_acc_val / len(val_ds): .3f} | Epoch Time: {epoch_time//60:.0f}m {epoch_time%60:.2f}s')

        # store results
        result = {'epoch': epoch_num + 1,
                  'train_loss': total_loss_train / len(train_ds),
                  'train_acc': total_acc_train / len(train_ds),
                  'val_loss': total_loss_val / len(val_ds),
                  'val_acc': total_acc_val / len(val_ds),
                  'epoch_time': epoch_time
                 }
        history.append(result)

    time_taken = perf_counter() - start
    print(f"\nTime Taken to train the model: {time_taken//60:.0f}m {time_taken%60:.2f}s")

    return history

In [25]:
history = train(model, train_loader, val_loader, 5e-5, 20)

  0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)



Epochs: 1/20 | Train Loss:  0.285 | Train Accuracy:  0.178 | Val Loss:  0.278 | Val Accuracy:  0.203 | Epoch Time: 1m 54.59s

Epochs: 2/20 | Train Loss:  0.279 | Train Accuracy:  0.189 | Val Loss:  0.278 | Val Accuracy:  0.203 | Epoch Time: 1m 57.26s

Epochs: 3/20 | Train Loss:  0.279 | Train Accuracy:  0.189 | Val Loss:  0.278 | Val Accuracy:  0.203 | Epoch Time: 1m 57.68s

Epochs: 4/20 | Train Loss:  0.279 | Train Accuracy:  0.189 | Val Loss:  0.278 | Val Accuracy:  0.203 | Epoch Time: 1m 57.34s

Epochs: 5/20 | Train Loss:  0.279 | Train Accuracy:  0.189 | Val Loss:  0.277 | Val Accuracy:  0.203 | Epoch Time: 1m 57.31s

Epochs: 6/20 | Train Loss:  0.279 | Train Accuracy:  0.189 | Val Loss:  0.278 | Val Accuracy:  0.203 | Epoch Time: 1m 57.39s

Epochs: 7/20 | Train Loss:  0.279 | Train Accuracy:  0.189 | Val Loss:  0.277 | Val Accuracy:  0.203 | Epoch Time: 1m 57.50s

Epochs: 8/20 | Train Loss:  0.279 | Train Accuracy:  0.189 | Val Loss:  0.277 | Val Accuracy:  0.204 | Epoch Time: 1m

In [ ]:
df.head()

In [ ]:
def predict(sentence):
    inputs = tokenizer(sentence, add_special_tokens=False, \
                                truncation=True, padding=True, \
                                return_tensors='pt')

    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    inputs['attention_mask'] = inputs['attention_mask'].to(device)
    inputs['input_ids'] = inputs['input_ids'].to(device)
    inputs.pop("token_type_ids", None)
    # inputs = {k: v.to(device) for k, v in inputs.items()}

    output1 = model(**inputs)
    preds1 = torch.argmax(output1, dim=1)

    # output2 = model(**inputs)
    # preds2 = torch.argmax(output2, dim=1)

    a1 = preds1.item()//3
    s1 = preds1.item()%3

    # a2 = preds2.item()//3
    # s2 = preds2.item()%3

    aspect_conversion_dict = {0: 'RESTAURANT#GENERAL', 1: 'RESTAURANT#PRICES', 2: 'RESTAURANT#MISCELLANEOUS', 3: 'FOOD#QUALITY',
    4: 'FOOD#PRICES', 5: 'FOOD#STYLE&OPTIONS', 6: 'DRINKS#QUALITY', 7: 'DRINKS#PRICES',
    8: 'DRINKS#STYLE&OPTIONS', 9: 'LOCATION#GENERAL', 10: 'AMBIENCE#GENERAL', 11: 'SERVICE#GENERAL'}
    sentiment_conversion_dict = {2: 'positive', 1: 'neutral', 0: 'negative'}

    aspect1 = aspect_conversion_dict[a1]
    sentiment1 = sentiment_conversion_dict[s1]

    # aspect2 = aspect_conversion_dict[a2]
    # sentiment2 = sentiment_conversion_dict[s2]

    prediction = {aspect1: sentiment1, }
                  # aspect2: sentiment2}

    return prediction #, a1, a2, s1, s2

In [ ]:
sentence = "Nhà hàng có không gian đẹp, nhưng chất lượng món ăn lại không như mong đợi, phục vụ hơi chậm."
# inputs = tokenizer(sentence)
# print(inputs)
prediction = predict(sentence)
print(prediction)

In [ ]:
test_df = pd.DataFrame(readData(config.test_path))
test_df.head()

In [ ]:
test_df['aspect1'] = test_df['aspect1'].apply(lambda x: aspect_conversion_dict[x])
test_df['sentiment1'] = test_df['sentiment1'].apply(lambda x: sentiment_conversion_dict[x])

test_df['aspect2'] = test_df['aspect2'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
test_df['sentiment2'] = test_df['sentiment2'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

test_df['aspect3'] = test_df['aspect3'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
test_df['sentiment3'] = test_df['sentiment3'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

test_df['aspect4'] = test_df['aspect4'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
test_df['sentiment4'] = test_df['sentiment4'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

# df['aspect5'] = df['aspect5'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
# df['sentiment5'] = df['sentiment5'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

a1 = test_df['aspect1']
s1 = test_df['sentiment1']
a1 += 1
s1 += 1
test_df['labels1'] = a1*s1 - 1

# a2 = test_df['aspect2']
# s2 = test_df['sentiment2']
# a2 += 1
# s2 += 1
# test_df['labels2'] = a2*s2 - 1

In [ ]:
# Hàm test
def test(data_loader):
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")

    predicts = []
    real_values = []

    for inputs, label in data_loader:
        inputs['input_ids'] = inputs['input_ids'].to(device)
        inputs.pop("token_type_ids", None)
        inputs['attention_mask'] = inputs['attention_mask'].to(device)

        outputs = model(**inputs)

        label = label.to(torch.long)
        label = label.to(device)

        # acc = get_accuracy(outputs, label)

        _, pred = torch.max(outputs, dim=1)
        predicts.extend(pred)
        real_values.extend(label)

    predicts = torch.stack(predicts).cpu()
    real_values = torch.stack(real_values).cpu()
    print(classification_report(real_values, predicts))
    return real_values, predicts

In [ ]:
data_test = ReviewsDataset(test_df, tokenizer, label_col='labels1')
test_loader = DataLoader(data_test, config.batch_size, shuffle=True,
                          num_workers=config.num_workers,
                          pin_memory=config.pin_memory)

In [ ]:
real_values, predicts = test(test_loader)

#Plotting results

In [ ]:
# train_losses = [x['train_loss'] for x in history]
# val_losses = [x['val_loss'] for x in history]
# epochs = [x['epoch'] for x in history]

# plt.locator_params(axis='x', nbins=5)
# plt.plot(epochs, train_losses, label='Train-Losses')
# plt.plot(epochs, val_losses, label='Validation-Losses')
# plt.xlabel('Epcohs')
# plt.ylabel('Losses')
# plt.title('Losses vs Epochs')
# plt.legend()

# # print(train_loss)

In [ ]:
def plot(history, name="HistoryPlot", figsize=(20, 9)):
    fig = plt.figure(figsize=figsize)
    epochs = [x['epoch'] for x in history]

    # Plotting Losses
    ax1 = fig.add_subplot(121)
    ax1.locator_params(axis='x', nbins=5)
    train_losses = [x['train_loss'] for x in history]
    val_losses = [x['val_loss'] for x in history]
    ax1.plot(epochs, train_losses, label='Train-Losses')
    ax1.plot(epochs, val_losses, label='Validation-Losses')
    plt.xlabel('Epochs')
    plt.ylabel('Losses')
    plt.title('Losses vs Epochs')
    plt.legend()

    # Plotting Accuracies
    ax2 = fig.add_subplot(122)
    ax2.locator_params(axis='x', nbins=5)
    train_accs = [x['train_acc'].cpu() for x in history]
    val_accs = [x['val_acc'].cpu() for x in history]
    ax2.plot(epochs, train_accs, label='Train-Accuracies')
    ax2.plot(epochs, val_accs, label='Validation-Accuracies')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracies')
    plt.title('Accuracies vs Epochs')
    plt.legend()

    fig.savefig('./'+name+".jpg")
    plt.show()

In [ ]:
plot(history, name="HistoryPlot1")